In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
%reload_ext autoreload

In [2]:
# Cell 1 (only one needed now)
from src import (
    extract_logon_features,
    extract_http_features,
    extract_email_features,
    fuse_feature_matrices,
    InsiderThreatDetector,
)

In [ ]:
# Cell 2 — extraction

hist_logon = extract_logon_features("data/r4.2/logon.csv", "2010-01-01", "2010-07-01", dev_mode=True)
hist_http  = extract_http_features("data/r4.2/http.csv",   "2010-01-01", "2010-07-01", dev_mode=True)
hist_email = extract_email_features("data/r4.2/email.csv", "2010-01-01", "2010-07-01", dev_mode=True)

live_logon = extract_logon_features("data/r4.2/logon.csv", "2011-04-01", None, dev_mode=True)
live_http  = extract_http_features("data/r4.2/http.csv",   "2011-04-01", None, dev_mode=True)
live_email = extract_email_features("data/r4.2/email.csv", "2011-04-01", None, dev_mode=True)

In [ ]:
import pandas as pd
df = pd.read_parquet(hist_logon)
print(df.columns.tolist())  # should include 'total_event_buckets' — the H5 rename
print(df.index.names)        # should be ['user', 'activity_date'''

In [ ]:
fused_hist = fuse_feature_matrices(
    hist_logon, hist_http, hist_email,
    output_prefix="historical_baseline",
)
fused_live = fuse_feature_matrices(
    live_logon, live_http, live_email,
    output_prefix="live_test",
)

In [ ]:
df = pd.read_parquet(fused_hist)
print(df.shape)
print(df.columns.tolist())  # should be all 15 features in MASTER_FEATURE_SCHEMA

In [3]:
import glob
import pandas as pd

hist_files = sorted(glob.glob("features/historical_baseline_*.parquet"))
live_files = sorted(glob.glob("features/live_test_*.parquet"))

print(f"Historical files found: {len(hist_files)}")
for f in hist_files:
    df = pd.read_parquet(f)
    print(f"  {f}: {len(df):,} rows")

print(f"\nLive files found: {len(live_files)}")
for f in live_files:
    df = pd.read_parquet(f)
    print(f"  {f}: {len(df):,} rows")

Historical files found: 1
  features\historical_baseline_20260516_001301.parquet: 126,892 rows

Live files found: 1
  features\live_test_20260516_001301.parquet: 27,038 rows


In [4]:
from src import InsiderThreatDetector, fuse_feature_matrices

# Point variables at the parquets that already exist on disk
import glob
hist_files = sorted(glob.glob("features/historical_baseline_*.parquet"))
live_files = sorted(glob.glob("features/live_test_*.parquet"))

fused_hist = hist_files[-1] if hist_files else None  # latest one
fused_live = live_files[-1] if live_files else None

print(f"Historical: {fused_hist}")
print(f"Live: {fused_live}")

Historical: features\historical_baseline_20260516_001301.parquet
Live: features\live_test_20260516_001301.parquet


In [17]:
# Cell 4 — training
detector = InsiderThreatDetector(mad_threshold=3.5, missing_data_tolerance=0.2)
pkl_path = detector.fit_baseline(fused_hist)
print(f"Trained model saved to: {pkl_path}")

2026-05-16 01:08:58,490 - INFO - TRAINING PHASE: Loading historical baseline from features\historical_baseline_20260516_001301.parquet
2026-05-16 01:08:58,552 - INFO - [fit_baseline] Input validation passed. Rows: 126892 | Numeric features: 15
2026-05-16 01:08:58,755 - INFO - [fit_baseline] MAD baseline computed | features=15 | zero_mad_cols=['total_event_buckets', 'off_hours_ratio', 'unique_systems_accessed', 'failed_login_ratio', 'active_orphan_sessions', 'keyword_match_indicator', 'unique_external_domains', 'after_hours_browsing', 'after_hours_emails', 'large_attachment_count']
2026-05-16 01:09:00,054 - INFO - [fit_baseline] ISO threshold (percentile=1.0) | threshold=-0.1342 | score_range=[-0.2209, 0.1405]
2026-05-16 01:09:00,067 - INFO - [fit_baseline] Training score percentiles stored | min=-0.2209 | p50=0.0848 | max=0.1405
2026-05-16 01:09:00,120 - INFO - [fit_baseline] Model serialized | version=20260516_010900 | path=iso_pipeline_v20260516_010900.pkl | features=['total_event_bu

Trained model saved to: iso_pipeline_v20260516_010900.pkl


In [18]:
import joblib
d = joblib.load(pkl_path)
print("is_fitted:           ", d.is_fitted)
print("features:            ", list(d.baseline_stats.keys()))
print("iso_threshold:       ", d.iso_threshold)
print("has train_percentiles:", hasattr(d, "train_score_percentiles"))
print("model_version:       ", d.model_version)

is_fitted:            True
features:             ['total_event_buckets', 'off_hours_ratio', 'unique_systems_accessed', 'failed_login_ratio', 'active_orphan_sessions', 'url_visits', 'upload_event_count', 'keyword_match_indicator', 'unique_external_domains', 'after_hours_browsing', 'emails_sent', 'external_recipients', 'after_hours_emails', 'attachments_sent', 'large_attachment_count']
iso_threshold:        -0.13419315596751505
has train_percentiles: True
model_version:        20260516_010900


In [19]:
import pandas as pd

df = pd.read_parquet("features/scored_features_latest.parquet")

print(f"Rows scored: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print()
print("Threat summary:")
print(f"  confirmed_threat : {df['confirmed_threat'].sum()}")
print(f"  high_risk_review : {df['high_risk_review'].sum()}")
print(f"  data_loss_ioc    : {df['data_loss_ioc'].sum()}")
print()
print("risk_score distribution (calibrated percentiles — C2):")
print(df['risk_score'].describe())
print()
print("Top 5 users by risk_score:")
print(df.nlargest(5, 'risk_score')[['user', 'activity_date', 'risk_score',
                                     'confirmed_threat', 'high_risk_review']])

Rows scored: 27,038
Columns: ['user', 'activity_date', 'data_quality_risk', 'mad_score_count', 'mad_critical_flag', 'iso_forest_raw_score', 'iso_forest_flag', 'confirmed_threat', 'high_risk_review', 'data_loss_ioc', 'risk_score', 'shap_total_event_buckets', 'shap_off_hours_ratio', 'shap_unique_systems_accessed', 'shap_failed_login_ratio', 'shap_active_orphan_sessions', 'shap_url_visits', 'shap_upload_event_count', 'shap_keyword_match_indicator', 'shap_unique_external_domains', 'shap_after_hours_browsing', 'shap_emails_sent', 'shap_external_recipients', 'shap_after_hours_emails', 'shap_attachments_sent', 'shap_large_attachment_count', 'total_event_buckets', 'off_hours_ratio', 'unique_systems_accessed', 'failed_login_ratio', 'active_orphan_sessions', 'url_visits', 'upload_event_count', 'keyword_match_indicator', 'unique_external_domains', 'after_hours_browsing', 'emails_sent', 'external_recipients', 'after_hours_emails', 'attachments_sent', 'large_attachment_count']

Threat summary:
  co

In [20]:
import pandas as pd

# 1. Confirm what fraction of training scores fall below the threshold
import joblib
det = joblib.load("iso_pipeline_v20260516_010900.pkl")
percentiles = det.train_score_percentiles
threshold = det.iso_threshold
below = sum(1 for p in percentiles if p < threshold) / len(percentiles)
print(f"Fraction of training distribution below ISO threshold: {below:.1%}")

# 2. Check the MAD-zero feature distributions in training data
import pandas as pd
df = pd.read_parquet("features/live_test_20260516_001301.parquet")
for col in ['total_event_buckets', 'off_hours_ratio', 'keyword_match_indicator',
            'after_hours_browsing', 'large_attachment_count']:
    print(f"{col}:")
    print(f"  median={df[col].median()}, zero_rate={(df[col]==0).mean():.1%}")

# 3. How many users flagged on a typical day
scored = pd.read_parquet("features/scored_features_latest.parquet")
print(f"\nMAD flag distribution:")
print(scored['mad_score_count'].describe())
print(f"\nISO flag rate: {scored['iso_forest_flag'].mean():.1%}")
print(f"MAD critical rate: {scored['mad_critical_flag'].mean():.1%}")

Fraction of training distribution below ISO threshold: 1.0%
total_event_buckets:
  median=2.0, zero_rate=0.0%
off_hours_ratio:
  median=0.0, zero_rate=57.5%
keyword_match_indicator:
  median=0.0, zero_rate=100.0%
after_hours_browsing:
  median=0.0, zero_rate=71.4%
large_attachment_count:
  median=0.0, zero_rate=100.0%

MAD flag distribution:
count    27038.000000
mean         0.187551
std          0.546263
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          4.000000
Name: mad_score_count, dtype: float64

ISO flag rate: 1.0%
MAD critical rate: 1.4%
